<a href="https://colab.research.google.com/github/Xrash09/CSA-AI-ML/blob/main/data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1-Data Understanding:

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"/content/PSComp.csv", comment="#")
print(df.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Rushil Shah\\Downloads\\csaaiml\\data\\PSComp.csv'

In [ ]:
pd.set_option("display.max_columns", None)
df.head()
df.dtypes

In [ ]:
missing_pct = df.isna().mean()*100
missing_pct = missing_pct.sort_values(ascending=False)
print(missing_pct)

In [ ]:
df_t = df[df["pl_eqt"].notna()].copy()
print(df_t.shape)

missing_t = (df_t.isna().mean() * 100).sort_values(ascending=False)
print(missing_t)

In [ ]:
feats =["pl_eqt", "pl_rade", "pl_bmasse", "pl_orbper", "pl_orbsmax", "st_teff", "st_rad", "st_mass", "sy_dist"]
df_t[feats].describe()

In [ ]:
#can a

# 2-Exploratory Data Analysis

### 2a-Distribution of Temperature(target)

In [ ]:
df_t["pl_eqt"].plot(kind="hist" , bins = 50 , title= "Distribution of Equilibrium Temperature")

In [ ]:
#calculating mean and median
eqt_mean = np.mean(df_t["pl_eqt"])
eqt_median = np.median(df_t["pl_eqt"])

print(f"For equlibrium temperature, mean: {eqt_mean:.1f} K and median: {eqt_median} K \n")

#finding hottest planet
print(df_t.nlargest(1, "pl_eqt")[["pl_name", "pl_eqt"]])

'pl_eqt' is right skewed: the middle 50% percent of planets lie between about 500K and 1100K
(median=758K, mean=836.4K) , with the hottest planet being 'KOI-55 b' with a temperature of 7777.78K.i

### 2b- Skewness of Orbital Period

In [ ]:
df_t["pl_orbper"].plot(kind="hist", bins=50, title="Orbital Period")

In [ ]:
np.log10(df_t["pl_orbper"]).plot(kind ="hist" , bins =50 , title="Orbital Period(log10)")

In [ ]:
missing_t = (df_t.isna().mean() * 100).sort_values(ascending=False)
print(missing_t)

### Extreme Temperatures: The Hottest Planets

**Observations**
- The five hottest planets all have very hot host stars (6,540-27,730 K) and tiny orbits (0.006-0.05 au). The top two, KOI-55 b and c, orbit a star at 27,730 K and reach
  7,000+ K.
- These are not data errors. They agree with the physics: temperature rises with star temperature and falls with distance, so a hot star plus a very close orbit gives an
  extreme value.
- They are far above the typical planet (median 758 K), so they are high-leverage points that can pull a least-squares fit toward them.

In [ ]:
df_t["pl_orbper"].plot(kind="hist", bins=50, title="pl_orbper raw")

In [ ]:
print(df_t.nlargest(5, "pl_eqt")[["pl_name", "pl_eqt", "st_teff", "pl_orbsmax"]])

In [ ]:
missing_t = (df_t[feats].isna().mean() * 100).sort_values(ascending=False)
missing_t.plot(kind="barh", title="% missing per feature")

### Distance vs Equilibrium Temperature:
equilibrium temperature falls off with orbital distance roughly as
T propotional to a^(-0.5), where a is the semi-major axis. I plotted the raw values first, then again after take log base 10.

In [ ]:
df_t.plot(kind="scatter", x="pl_orbsmax", y="pl_eqt", alpha=0.3, figsize=(8, 5), title="Raw: distance vs temperature")

In [ ]:
plot_df = pd.DataFrame({
    "log_dist": np.log10(df_t["pl_orbsmax"]),
    "log_temp": np.log10(df_t["pl_eqt"]),
})
plot_df.plot(kind="scatter", x="log_dist", y="log_temp", alpha=0.3, figsize=(8, 5),
             title="Log-log: distance vs temperature")

d = plot_df.dropna()
slope, intercept = np.polyfit(d["log_dist"], d["log_temp"], 1)
print("slope:", slope, "intercept:", intercept)

**Observations**
- **Raw scale:** Nearly all planets sit at very small distances, stacked in a narrow
  column, while a few lie beyond 1000 au. Temperature spans 0-3000 K at almost the same
  distance, so no straight line on raw values could fit this well.
- **Log-log scale:** Most planets form a tight band sloping downward, as the physics
  predicts, so log distance is a far better feature. Planets beyond ~10 au scatter away
  from the band, including some near 0 K and some that are hot.

In [ ]:
df_t["pl_eqt"].plot(kind="hist", bins=50, title="Distribution of equilibrium temperature")

In [ ]:
df_t.plot(kind="scatter", x="st_teff", y="pl_eqt", alpha=0.3, figsize=(8, 5),
          title="Stellar temperature vs equilibrium temperature")

In [ ]:
cols = ["pl_eqt", "pl_rade", "pl_bmasse", "pl_orbper", "pl_orbsmax",
        "st_teff", "st_rad", "st_mass", "sy_dist"]
df_t[cols].corr()["pl_eqt"].sort_values(ascending=False)

In [ ]:
logged = df_t[cols].copy()
for c in ["pl_bmasse", "pl_orbper", "pl_orbsmax", "sy_dist", "pl_rade"]:
    logged[c] = np.log10(logged[c])
logged.corr()["pl_eqt"].sort_values(ascending=False)